In [ ]:
%env AWS_PROFILE=platform-developer

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import boto3
from botocore.config import Config

ADAPTER_TABLE = "vhs-sierra-sierra-adapter-20200604"
ADAPTER_BUCKET = "wellcomecollection-vhs-sierra-sierra-adapter-20200604"

OUTPUT_PATH = Path("data/sierra-raw-works.parquet")
CHUNK_SIZE = 25_000
WORKERS = 32

# One shared client per service: boto3 clients are thread-safe, Sessions are not.
config = Config(max_pool_connections=WORKERS * 2)
dynamodb = boto3.client("dynamodb", config=config)
s3 = boto3.client("s3", config=config)

In [ ]:
SEGMENTS = 24


def scan_segment(segment: int) -> list[tuple[str, str]]:
    records = []
    kwargs = {
        "TableName": ADAPTER_TABLE,
        "Segment": segment,
        "TotalSegments": SEGMENTS,
        "ProjectionExpression": "id, payload",
    }

    while True:
        response = dynamodb.scan(**kwargs)

        for item in response["Items"]:
            # Older rows in this store spell the pointer `location` rather than `payload`.
            pointer = item.get("payload") or item.get("location")
            records.append((item["id"]["S"], pointer["M"]["key"]["S"]))

        if "LastEvaluatedKey" not in response:
            return records

        kwargs["ExclusiveStartKey"] = response["LastEvaluatedKey"]


with ThreadPoolExecutor(max_workers=SEGMENTS) as executor:
    segments = executor.map(scan_segment, range(SEGMENTS))
    manifest = sorted(record for segment in segments for record in segment)

print(f"{len(manifest):,} records to download")
manifest[:3]

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

SCHEMA = pa.schema([("id", pa.string()), ("body", pa.string())])

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def download_body(key: str) -> str:
    return s3.get_object(Bucket=ADAPTER_BUCKET, Key=key)["Body"].read().decode("utf-8")


with (
    pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression="zstd") as writer,
    ThreadPoolExecutor(max_workers=WORKERS) as executor,
):
    for start in range(0, len(manifest), CHUNK_SIZE):
        chunk = manifest[start : start + CHUNK_SIZE]
        bodies = executor.map(download_body, [key for _, key in chunk])

        writer.write_table(
            pa.table([[id for id, _ in chunk], list(bodies)], schema=SCHEMA)
        )
        print(f"{start + len(chunk):>9,} / {len(manifest):,}")

print(f"Wrote {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1e9:.1f} GB)")

In [ ]:
import polars as pl

works = pl.scan_parquet(OUTPUT_PATH)

print(f"{works.select(pl.len()).collect().item():,} records")
works.head(3).collect()